# Lab 1 — Advanced RAG: from a basic retriever to production
**Block 1 · Advanced RAG & MCP · PGE5 / M2**

Guillaume's lab gave you a pure-Python TF-IDF RAG — elegant, zero-dependency, perfect for
understanding the principle. This lab goes further: we build the pipeline you would deploy
in production, measuring the improvement at each step.

### Objectives
1. **Measure** your existing retriever — get a number before touching anything.
2. **Parent-child chunking** — retrieve small, return large.
3. **Hybrid search** — BM25 (exact terms) + dense (semantic) + RRF (fusion).
4. **Cross-encoder reranking** — final precision before injecting into the LLM.
5. **Understand the impact** of each improvement on retrieval quality.

> **Offline mode.** All demo cells run without an API key thanks to the `MockLLMClient`.
> As soon as a key is present in `.env`, the same cells call the real model — no change needed.
> This lab has **no external dependencies in offline mode** (no Chroma, no Ollama; the
> optional real-RAGAS cell 6b self-installs pinned `ragas`/`langchain` only when run online).


## 0. Setup

In [1]:
from llm_helpers import (
    make_client, credentials_available, LLMClient, MockLLMClient,
    ToolRegistry, tool_schema, run_agent,
)

ONLINE = credentials_available()
print("Mode:", "🌐 ONLINE (real model)" if ONLINE else "⚙️  OFFLINE (mock)")

def demo(script=None):
    "Fresh client for a demo cell: real model if a key is available, otherwise scripted mock."
    return make_client(offline_script=script, quiet=True)

# ── Demo corpus ───────────────────────────────────────────────────────────
# 12 documents about climate displacement. IMPORTANT: the corpus deliberately
# contains DISTRACTORS — documents that share vocabulary with the questions but
# do NOT hold the answer. This is what lets you see the difference between a naive
# retriever and the production pipeline.
# TODO: replace with your own domain (keep the distractors!).
CORPUS = {
    # ── Documents holding the real answers ──
    "unhcr_2023":
        "According to UNHCR, 21.5 million people are displaced every year by sudden "
        "climate events: floods, storms and droughts. "
        "This figure has risen 25% compared with the previous decade.",
    "worldbank_2050":
        "The World Bank estimates that by 2050, 216 million people could be forced to "
        "migrate within their own country because of climate change, "
        "including 40 million in South Asia.",
    "idmc_philippines":
        "The IDMC reports that the Philippines saw a 15% increase in typhoon-related "
        "displacement over the last five years, one of the highest rates in the world.",
    "mekong_delta":
        "The Mekong Delta in Vietnam risks losing up to 40% of its floodable farmland "
        "by 2100 due to rising sea levels and land subsidence.",
    "southeast_asia_exposure":
        "Bangladesh, Vietnam and the Philippines are among the countries most exposed "
        "to climate displacement in Southeast Asia, due to their long low-lying coastlines "
        "and dense coastal populations.",

    # ── Distractors: same theme, close vocabulary, but NOT the answer ──
    "climate_finance":
        "The Green Climate Fund has mobilised 10 billion dollars to help developing "
        "countries. Adaptation finance nonetheless remains far below estimated needs.",
    "cop_negotiations":
        "COP negotiations on loss and damage led to the creation of a dedicated fund. "
        "Vulnerable countries have demanded binding financial commitments for years.",
    "refugee_definition":
        "The 1951 Geneva Convention does not recognise 'climate refugees' as a legal "
        "status. The correct term is 'internally displaced persons' when borders are not crossed.",
    "drought_agriculture":
        "Prolonged drought in the Horn of Africa has destroyed crops and livestock. "
        "Food insecurity affects millions of people, worsening regional tensions.",
    "sea_level_cities":
        "Coastal cities such as Jakarta, Miami and Alexandria are investing in dikes and "
        "pumping systems. Jakarta is sinking several centimetres per year under urban weight.",
    "typhoon_intensity":
        "Scientists observe that Pacific typhoons are gaining intensity, even though their "
        "total frequency stays stable. A warmer ocean supplies more energy to storms.",
    "economic_migration":
        "Voluntary economic migration differs from forced displacement. Millions of workers "
        "leave rural areas for Asian megacities for employment reasons, not climate.",
}
print(f"{len(CORPUS)} documents loaded (5 with answers, 7 distractors).")


Mode: 🌐 ONLINE (real model)
12 documents loaded (5 with answers, 7 distractors).


## 1. The basic retriever — pure-Python TF-IDF

Before adding any dependency, we start from the same TF-IDF as Guillaume: zero install,
runs everywhere. This is your **baseline** that the metric will measure.


In [2]:
import math, re
from collections import Counter

def _tokenise(text):
    return re.findall(r"[a-zàâçéèêëîïôûùüÿœ0-9]+", text.lower())

class TinyTfidf:
    "Minimal TF-IDF retriever (cosine), no external dependency."
    def __init__(self, docs: dict):
        self.names = list(docs.keys())
        self.texts = list(docs.values())
        tok = [_tokenise(t) for t in self.texts]
        N   = len(self.texts)
        df  = Counter(w for ts in tok for w in set(ts))
        self.idf  = {w: math.log((1 + N) / (1 + df[w])) + 1 for w in df}
        self.vecs = [self._vec(ts) for ts in tok]

    def _vec(self, tokens):
        tf = Counter(tokens); n = len(tokens) or 1
        return {w: tf[w] / n * self.idf.get(w, 0.0) for w in tf}

    @staticmethod
    def _cos(a, b):
        num = sum(a[w] * b[w] for w in set(a) & set(b))
        na  = math.sqrt(sum(v**2 for v in a.values()))
        nb  = math.sqrt(sum(v**2 for v in b.values()))
        return num / (na * nb) if na * nb else 0.0

    def search(self, query: str, k: int = 3) -> list:
        qv  = self._vec(_tokenise(query))
        scores = [(self._cos(qv, v), t) for v, t in zip(self.vecs, self.texts)]
        return sorted(scores, reverse=True)[:k]

retriever_base = TinyTfidf(CORPUS)

q = "Which Southeast Asian countries are the most exposed?"
print(f"Query: {q!r}")
for score, doc in retriever_base.search(q, k=2):
    print(f"  [{score:.3f}] {doc[:100]}…")


Query: 'Which Southeast Asian countries are the most exposed?'
  [0.422] Bangladesh, Vietnam and the Philippines are among the countries most exposed to climate displacement…
  [0.100] Voluntary economic migration differs from forced displacement. Millions of workers leave rural areas…


## 2. Measuring retrieval quality — the baseline

> **Golden rule:** do not touch anything before you have a number. You cannot improve
> what you do not measure.

RAGAS is the standard tool. If it is not installed, we use a rank-sensitive pure-Python
proxy: **does the right document reach the top?** (A simple word overlap is not enough —
the distractors share vocabulary.)


In [3]:
# RAGAS is the standard tool. If not installed, we use a rank-sensitive pure-Python
# proxy: does the RIGHT document reach the top?
# (word overlap alone is not enough — distractors share vocabulary.)

QUESTIONS = [
    "How many people are displaced every year by sudden disasters?",
    "Which countries are most exposed along Southeast Asian coastlines?",
    "How many could be forced to migrate within their country by 2050?",
    "How much did typhoon displacement increase in the Philippines?",
    "What share of farmland could the delta lose by 2100?",
]
GOLD_DOC = [
    "unhcr_2023",
    "southeast_asia_exposure",
    "worldbank_2050",
    "idmc_philippines",
    "mekong_delta",
]
GROUND_TRUTH = [
    "21.5 million people displaced every year according to UNHCR",
    "Bangladesh, Vietnam and the Philippines",
    "216 million people could migrate within their own country",
    "a 15% increase in typhoon-related displacement",
    "losing up to 40% of its floodable farmland",
]

def _tokset(t):
    return set(re.findall(r"[a-zàâçéèêëîïôûù0-9]+", t.lower()))

def hit_at_k(retriever_fn, k=3):
    """Fraction of questions whose GOLD document appears in the top-k.
    Distractor-sensitive: a retriever that surfaces a distractor instead of the
    right document is penalised, even if the distractor shares vocabulary."""
    hits = 0
    for q, gold_id in zip(QUESTIONS, GOLD_DOC):
        chunks = retriever_fn(q, k)          # list of texts
        gold_tok = _tokset(CORPUS[gold_id])
        found = any(len(_tokset(c) & gold_tok) / max(1, len(gold_tok)) > 0.5 for c in chunks)
        hits += found
    return round(hits / len(QUESTIONS), 3)

def mrr(retriever_fn, k=5):
    """Mean Reciprocal Rank: rewards having the right document AT THE TOP.
    1.0 = right doc in position 1; 0.5 = position 2; etc."""
    total = 0.0
    for q, gold_id in zip(QUESTIONS, GOLD_DOC):
        chunks = retriever_fn(q, k)
        gold_tok = _tokset(CORPUS[gold_id])
        rank = 0
        for pos, c in enumerate(chunks, start=1):
            if len(_tokset(c) & gold_tok) / max(1, len(gold_tok)) > 0.5:
                rank = pos
                break
        total += (1.0 / rank) if rank else 0.0
    return round(total / len(QUESTIONS), 3)

print("Metrics ready: hit@k (right doc in top-k) and MRR (right doc at the top).")
print("With 7 distractors in the corpus, a naive retriever will miss some.")


Metrics ready: hit@k (right doc in top-k) and MRR (right doc at the top).
With 7 distractors in the corpus, a naive retriever will miss some.


## 3. Parent-Child Chunking

**Problem:** with fixed-size chunks, a key fact can be cut in two.
Neither chunk A nor chunk B holds the complete information.

**Solution:** index small child chunks (precision) but return the large parent
chunk in the prompt (full context).


In [4]:
# Parent-child chunking — pure Python, no Chroma

def split_words(text: str, size: int, overlap: int) -> list:
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunks.append(" ".join(words[i:i+size]))
        i += size - overlap
    return chunks

# Two-level index
children = {}   # chunk_id -> short text (200 words)
parents  = {}   # chunk_id -> long text  (800 words)
child_to_parent = {}

for doc_id, text in CORPUS.items():
    parent_chunks = split_words(text, size=80, overlap=10)   # ~800 words in prod
    for p_idx, parent_text in enumerate(parent_chunks):
        parent_id = f"{doc_id}_p{p_idx}"
        parents[parent_id] = parent_text
        child_chunks = split_words(parent_text, size=20, overlap=3)  # ~200 words in prod
        for c_idx, child_text in enumerate(child_chunks):
            child_id = f"{parent_id}_c{c_idx}"
            children[child_id] = child_text
            child_to_parent[child_id] = parent_id

# Retriever on children -> returns parents
retriever_children = TinyTfidf(children)

def retrieve_with_parent(query: str, k: int = 3) -> list:
    """Retrieve the nearest children, return their parents."""
    seen, result = set(), []
    for _, child_text in retriever_children.search(query, k=k*2):
        child_id = next((cid for cid, ct in children.items() if ct == child_text), None)
        if child_id:
            parent_id = child_to_parent[child_id]
            if parent_id not in seen:
                seen.add(parent_id)
                result.append(parents[parent_id])
    return result[:k]

q = QUESTIONS[0]
ctx = retrieve_with_parent(q, k=2)
print(f"Query: {q[:60]}…")
print(f"Returned context ({len(ctx)} blocks):")
for i, c in enumerate(ctx):
    print(f"  [{i+1}] {c[:120]}…")


Query: How many people are displaced every year by sudden disasters…
Returned context (2 blocks):
  [1] According to UNHCR, 21.5 million people are displaced every year by sudden climate events: floods, storms and droughts. …
  [2] The 1951 Geneva Convention does not recognise 'climate refugees' as a legal status. The correct term is 'internally disp…


## 4. Hybrid Search: BM25 + TF-IDF + RRF

**Problem:** dense search (TF-IDF / vectors) misses exact terms — acronyms, numbers,
proper nouns. BM25 catches them.

**Solution:** fuse both lists with **Reciprocal Rank Fusion** (RRF), without normalising
the scores — we work directly on ranks.

```
RRF_score(doc) = Σ  1 / (K + rank_i(doc))
              systems
```


In [5]:
# Pure-Python BM25 — same philosophy as TinyTfidf
class TinyBM25:
    "Minimal BM25-Okapi, no external dependency."
    def __init__(self, docs: dict, k1: float = 1.5, b: float = 0.75):
        self.names = list(docs.keys())
        self.texts = list(docs.values())
        self.tok   = [_tokenise(t) for t in self.texts]
        N      = len(self.texts)
        avgdl  = sum(len(ts) for ts in self.tok) / N
        df     = Counter(w for ts in self.tok for w in set(ts))
        self.idf = {w: math.log((N - df[w] + 0.5) / (df[w] + 0.5) + 1) for w in df}
        self._k1 = k1; self._b = b; self._avgdl = avgdl

    def _score(self, tok_q, idx):
        dl = len(self.tok[idx])
        sc = 0.0
        tf_d = Counter(self.tok[idx])
        for w in tok_q:
            if w not in self.idf: continue
            f = tf_d[w]
            num = self.idf[w] * f * (self._k1 + 1)
            den = f + self._k1 * (1 - self._b + self._b * dl / self._avgdl)
            sc += num / den
        return sc

    def search(self, query: str, k: int = 3) -> list:
        q = _tokenise(query)
        ranked = sorted(range(len(self.texts)),
                        key=lambda i: self._score(q, i), reverse=True)
        return [(self._score(q, i), self.texts[i]) for i in ranked[:k]]

def rrf_fusion(lists: list, K: int = 60) -> list:
    """Reciprocal Rank Fusion — fuses several lists of (score, text)."""
    scores = {}
    for lst in lists:
        for rank, (_, text) in enumerate(lst):
            scores[text] = scores.get(text, 0.0) + 1.0 / (K + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

# Hybrid retriever
bm25_retriever = TinyBM25(CORPUS)
dense_retriever = TinyTfidf(CORPUS)

def hybrid_retrieve(query: str, k: int = 5) -> list:
    dense_results = dense_retriever.search(query, k=k*2)
    bm25_results  = bm25_retriever.search(query, k=k*2)
    fused = rrf_fusion([dense_results, bm25_results])
    return [text for text, _ in fused[:k]]

q = QUESTIONS[1]
print(f"Query: {q!r}")
print("\nDense only:")
for s, d in dense_retriever.search(q, 2): print(f"  [{s:.3f}] {d[:80]}…")
print("\nBM25 only:")
for s, d in bm25_retriever.search(q, 2):  print(f"  [{s:.3f}] {d[:80]}…")
print("\nRRF fused:")
for i, d in enumerate(hybrid_retrieve(q, k=2)): print(f"  [{i+1}] {d[:80]}…")


Query: 'Which countries are most exposed along Southeast Asian coastlines?'

Dense only:
  [0.448] Bangladesh, Vietnam and the Philippines are among the countries most exposed to …
  [0.093] Voluntary economic migration differs from forced displacement. Millions of worke…

BM25 only:
  [10.191] Bangladesh, Vietnam and the Philippines are among the countries most exposed to …
  [2.314] Voluntary economic migration differs from forced displacement. Millions of worke…

RRF fused:
  [1] Bangladesh, Vietnam and the Philippines are among the countries most exposed to …
  [2] Voluntary economic migration differs from forced displacement. Millions of worke…


## 5. Cross-Encoder Reranking

TF-IDF and BM25 encode the **query** and the **document** separately.
A **cross-encoder** reads them *together* — it captures the fine-grained interactions
between the question and the passage. Result: better precision, but slower
(so we only apply it to the top-k candidates).

In production: `cross-encoder/ms-marco-MiniLM-L-6-v2` (sentence-transformers).
Here: a **simulated** cross-encoder that illustrates the idea without a dependency.


In [6]:
def cross_encoder_score(query: str, document: str) -> float:
    """
    Simulated cross-encoder — in production: sentence_transformers.CrossEncoder.
    Approximates relevance by combining term coverage and shared length.
    Replace with: reranker.predict([(query, doc)]) using the real model.
    """
    q_tok = set(_tokenise(query))
    d_tok = set(_tokenise(document))
    if not q_tok or not d_tok: return 0.0
    overlap   = len(q_tok & d_tok) / len(q_tok)
    doc_bonus = min(1.0, len(d_tok) / 50)   # richer-context bonus
    return round(overlap * 0.7 + doc_bonus * 0.3, 4)

def rerank(query: str, candidates: list, top_k: int = 3) -> list:
    """Rerank `candidates` by cross-relevance with the query."""
    scored = [(cross_encoder_score(query, doc), doc) for doc in candidates]
    return [doc for _, doc in sorted(scored, reverse=True)[:top_k]]

def production_retrieve(query: str, k_final: int = 3) -> list:
    """Full pipeline: hybrid -> parent-child -> rerank."""
    # 1. Hybrid over children
    dense_c = retriever_children.search(query, k=10)
    bm25_c  = TinyBM25(children).search(query, k=10)
    fused_c = rrf_fusion([dense_c, bm25_c])
    # 2. Go back up to parents
    seen, candidates = set(), []
    for text, _ in fused_c:
        child_id = next((cid for cid, ct in children.items() if ct == text), None)
        if child_id:
            pid = child_to_parent[child_id]
            if pid not in seen:
                seen.add(pid); candidates.append(parents[pid])
    # 3. Rerank
    return rerank(query, candidates, top_k=k_final)

q = QUESTIONS[2]
print(f"Query: {q!r}")
print("\nProduction pipeline:")
for i, d in enumerate(production_retrieve(q)):
    print(f"  [{i+1}] score={cross_encoder_score(q,d):.3f}  {d[:100]}…")


Query: 'How many could be forced to migrate within their country by 2050?'

Production pipeline:
  [1] score=0.751  The World Bank estimates that by 2050, 216 million people could be forced to migrate within their ow…
  [2] score=0.291  According to UNHCR, 21.5 million people are displaced every year by sudden climate events: floods, s…
  [3] score=0.267  Bangladesh, Vietnam and the Philippines are among the countries most exposed to climate displacement…


## 6. Comparison — did each fix earn its place?

In [7]:
# Retriever comparison — rank-sensitive metrics
# hit@3: is the right document in the top-3?   MRR: is it at the top?

def r_baseline(q, k=3):
    return [txt for _, txt in retriever_base.search(q, k)]

def r_hybrid(q, k=3):
    return hybrid_retrieve(q, k)

def r_production(q, k=3):
    return production_retrieve(q, k_final=k)

print("=== RETRIEVER COMPARISON (12-document corpus) ===\n")
print(f'{"Retriever":<32}{"hit@3":>8}{"MRR":>8}')
print("-" * 48)
for name, fn in [
    ("Baseline (TF-IDF)",            r_baseline),
    ("Hybrid (BM25 + TF-IDF + RRF)", r_hybrid),
    ("Production (hybrid + rerank)", r_production),
]:
    print(f"{name:<32}{hit_at_k(fn):>8}{mrr(fn):>8}")

print()
print("IMPORTANT — reading this result honestly:")
print("On a small, clean 12-document corpus, all three retrievers score well.")
print("That is expected: with so few documents and distinctive vocabulary,")
print("even plain TF-IDF finds the right passage. The gap between techniques")
print("ONLY appears at scale — hundreds of documents, ambiguous queries,")
print("near-duplicate distractors.")
print()
print("This is itself the lesson: DO NOT trust a retrieval improvement measured")
print("on toy data. To see hybrid search and reranking earn their keep, run")
print("RAGAS (see the B1 guide) on YOUR real corpus in the homework — that is")
print("where context_recall and context_precision reveal the difference.")


=== RETRIEVER COMPARISON (12-document corpus) ===

Retriever                          hit@3     MRR
------------------------------------------------
Baseline (TF-IDF)                    1.0     1.0
Hybrid (BM25 + TF-IDF + RRF)         1.0     1.0
Production (hybrid + rerank)         1.0     1.0

IMPORTANT — reading this result honestly:
On a small, clean 12-document corpus, all three retrievers score well.
That is expected: with so few documents and distinctive vocabulary,
even plain TF-IDF finds the right passage. The gap between techniques
ONLY appears at scale — hundreds of documents, ambiguous queries,
near-duplicate distractors.

This is itself the lesson: DO NOT trust a retrieval improvement measured
on toy data. To see hybrid search and reranking earn their keep, run
RAGAS (see the B1 guide) on YOUR real corpus in the homework — that is
where context_recall and context_precision reveal the difference.


### 6b. Real RAGAS (optional — runs online)

The hit@3 / MRR proxy above runs offline and needs no API key. But the **industry-standard**
tool is RAGAS, which scores four metrics — `context_recall`, `context_precision`,
`faithfulness`, `answer_relevancy` — the ones on the slide.

RAGAS uses an LLM to compute the scores, so it needs a key and the `ragas` package. The cell
below runs it **only if you are online**; otherwise it skips cleanly. In your homework, run
RAGAS on your real corpus — that is where these four numbers become meaningful.

In [8]:
# 6b — Real RAGAS on the 5 questions (optional; online only, self-contained).
# Everything needed is in this cell: it pins compatible versions, points RAGAS at
# YOUR provider (Groq/OpenAI/etc. via the same env vars llm_helpers uses), and if
# anything fails it falls back to the hit@3/MRR proxy above. No manual fixes needed.

import os, sys, subprocess

if not ONLINE:
    print("OFFLINE — skipping real RAGAS. The hit@3/MRR proxy above is your measurement.")
    print("Run this cell online (with a key in .env) to get the 4 slide metrics.")
else:
    try:
        # 1. Pinned, known-compatible versions (avoids the langchain_community break)
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "ragas==0.1.21", "langchain==0.2.16", "langchain-community==0.2.16",
             "langchain-openai==0.1.25", "datasets"],
            check=True,
        )

        from datasets import Dataset
        from ragas import evaluate
        from ragas.metrics import (context_recall, context_precision,
                                    faithfulness, answer_relevancy)
        from langchain_openai import ChatOpenAI, OpenAIEmbeddings

        # 2. Point RAGAS at the SAME endpoint you already use (Groq/OpenAI/etc.)
        base_url = os.getenv("OPENAI_BASE_URL")           # set for Groq users
        api_key  = os.getenv("OPENAI_API_KEY")
        model    = os.getenv("LLM_MODEL", "gpt-4o-mini")

        judge = ChatOpenAI(model=model, api_key=api_key, base_url=base_url,
                           temperature=0, max_retries=10)  # backoff on free-tier 429s
        print(f"RAGAS judge: model={model!r} base_url={base_url!r}")
        # Embeddings: Groq has no embeddings endpoint, so answer_relevancy (which needs
        # them) may be skipped there. The other 3 metrics work with the chat model alone.
        emb = None
        if not base_url or "openai.com" in (base_url or ""):
            emb = OpenAIEmbeddings(api_key=api_key)        # real OpenAI only

        # 3. Build the eval dataset from the production pipeline
        rows = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
        for q, gt in zip(QUESTIONS, GROUND_TRUTH):
            ctx = production_retrieve(q, k_final=3)
            ans = ctx[0] if ctx else ""
            rows["question"].append(q)
            rows["answer"].append(ans)
            rows["contexts"].append(ctx)
            rows["ground_truth"].append(gt)
        ds = Dataset.from_dict(rows)

        # 4. Pick metrics that work with the available components
        metrics = [context_recall, context_precision, faithfulness]
        if emb is not None:
            metrics.append(answer_relevancy)   # needs embeddings

        kw = {"llm": judge}
        if emb is not None:
            kw["embeddings"] = emb
        try:
            from ragas.run_config import RunConfig
            kw["run_config"] = RunConfig(max_workers=1, timeout=180)  # sequential: free tiers allow ~1 req/s
        except ImportError:
            pass
        scores = evaluate(ds, metrics=metrics, **kw)

        print("=== RAGAS (the slide metrics) ===")
        print(scores)
        print("\nWrite these numbers down — compare after the fixes.")
        if emb is None:
            print("\nNote: answer_relevancy needs an embeddings endpoint (real OpenAI).")
            print("On OpenAI-compatible endpoints without embeddings (Groq, Mistral, ...)")
            print("the other three metrics are computed; that is expected.")

    except Exception as e:
        print(f"RAGAS did not run here ({type(e).__name__}).")
        print("→ Use the hit@3/MRR proxy above — it teaches the same before/after idea.")
        print("→ You can run full RAGAS on your real corpus in the homework.")



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: /Users/abdu07/Desktop/PGE5/AI-AGENTS-CS/part2/.venv/bin/python -m pip install --upgrade pip


/Users/abdu07/Desktop/PGE5/AI-AGENTS-CS/part2/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RAGAS judge: model='mistral-large-latest' base_url='https://api.mistral.ai/v1'


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]

Evaluating:   7%|▋         | 1/15 [00:05<01:18,  5.64s/it]

...

Evaluating:  87%|████████▋ | 13/15 [05:50<01:11, 35.92s/it]

Evaluating:  93%|█████████▎| 14/15 [05:51<00:25, 25.45s/it]

Evaluating: 100%|██████████| 15/15 [07:03<00:00, 39.36s/it]

Evaluating: 100%|██████████| 15/15 [07:03<00:00, 28.22s/it]

=== RAGAS (the slide metrics) ===
{'context_recall': 1.0000, 'context_precision': 1.0000, 'faithfulness': 1.0000}

Write these numbers down — compare after the fixes.

Note: answer_relevancy needs an embeddings endpoint (real OpenAI).
On OpenAI-compatible endpoints without embeddings (Groq, Mistral, ...)
the other three metrics are computed; that is expected.


## 7. The retriever as an **agent tool**

In [9]:
# We expose production_retrieve as a tool in run_agent 

def search_knowledge(query: str) -> str:
    "Search the internal document base for relevant passages."
    results = production_retrieve(query, k_final=3)
    if not results: return "No results found."
    return "\n---\n".join(results)

registry = ToolRegistry()
registry.register(
    tool_schema("search_knowledge",
                "Search the internal document base for relevant passages. "
                "Use FIRST before any other source. "
                "Do NOT use for calculations or pure-logic questions.",
                {"query": {"type": "string", "description": "Search query."}},
                ["query"]),
    search_knowledge,
)

client = demo([
    {"tool": "search_knowledge", "arguments": {"query": "climate displacement Asia"}},
    {"final": "Based on internal sources: Southeast Asia, notably Bangladesh and the "
              "Philippines, is among the most exposed regions."},
])
hist = run_agent(client, registry,
                 [{"role": "user",
                   "content": "Which Asian countries are most exposed to climate displacement?"}])
print("\nUsage:", client.usage_report())


[étape 1] outil: search_knowledge({'query': 'Asian countries most exposed to climate displacement'})
          ↳ Bangladesh, Vietnam and the Philippines are among the countries most exposed to climate displacement in Southeast Asia, due to their long low-lying coastlines and dense coastal populations.
---
The Green Climate Fund has mobilised 10 billion dollars to help developing countries. Adaptation finance nonetheless remains far below estimated needs.
---
Voluntary economic migration differs from forced displacement. Millions of workers leave rural areas for Asian megacities for employment reasons, not climate.


[étape 2] réponse finale → Here are some of the **Asian countries most exposed to climate displacement**, based on their vulnerability to climate risks such as rising sea levels, extreme weather events, and environmental degradation:

1. **Bangladesh**
   - Bangladesh is one of the most climate-vulnerable countries in the world. It faces severe risks from **flooding, cyclones, riverbank erosion, and sea-level rise**. Millions of people, particularly in low-lying coastal areas, are at risk of displacement.

2. **Vietnam**
   - Vietnam's long coastline and densely populated delta regions, such as the **Mekong Delta**, make it highly vulnerable to **sea-level rise, saltwater intrusion, and flooding**. These factors threaten agriculture and livelihoods, leading to potential displacement.

3. **Philippines**
   - The Philippines is frequently hit by **typhoons, storm surges, and flooding**, which displace thousands of people annually. Its geographic location and high population density in c

### 📝 Questions on the output above

Your exact output depends on which model you ran (Groq, OpenAI, mock…) and can even
change between runs. That's the point — answer these by **inspecting the output *you*
actually got**, not a fixed "correct" result.

**Q1 — Count the tool calls.** How many times did the agent call `search_knowledge` in
your run? Why might an agent search more than once before answering — and why might it
search only once? What would make *your* run search again?

**Q2 — Retrieved vs. used.** Look at the chunks your retriever returned. Some are relevant,
some are distractors. Did the final answer rely only on the relevant ones? How can you
*tell* from the output — and how could you tell more reliably than by eye?

**Q3 — Faithfulness check (do this for your own answer).** Take each factual claim in your
agent's final answer and try to locate it in the `CORPUS` dictionary (cell 2).
- Are there any claims that are **not** traceable to a retrieved document?
- If yes, where did they come from? If no, what in the setup kept the answer grounded?
- Which RAGAS metric measures exactly this, and which Block 3 technique reduces it?

*(This is the key skill: verifying grounding works on ANY model's output.)*

**Q4 — Read the usage line.** Note your own `calls`, `tokens in`, `tokens out`, and cost.
Why is "tokens in" usually much larger than "tokens out" in a RAG agent? If you switched to
a bigger/more expensive model, which number would grow fastest?

**Q5 — Reproducibility.** Run this cell **twice more**. Does the output change? Does the
final *answer* change even when the wording differs? What does that tell you about
evaluating an agent — can you grade it on one run, or do you need a fixed test set?

> *Discuss in pairs (5 min), then we compare across the different models people used —
> the differences themselves are the lesson.*

### Answers (from our run — live, mistral-large-latest)

**Q1.** One call to `search_knowledge`, with the query `'Asian countries most exposed to
climate displacement'` (the model's rephrasing of the user question). One search was enough
because the first retrieval already contained the answer. It would search again if the
retrieval came back empty or off-topic, or if the question had several parts needing
separate lookups.

**Q2.** Three chunks came back: `southeast_asia_exposure` (the right one, rank 1), then
`climate_finance` and `economic_migration`, both distractors. The answer's grounded core
(Bangladesh, Vietnam, Philippines, low-lying coastlines) comes from the rank-1 chunk.
Neither distractor contributes real content — at most, the closing line about "climate
adaptation and resilience-building" loosely echoes the theme of the `climate_finance`
chunk. Which is the point: judging this by eye is unreliable, and it also misses the
bigger problem in Q3. The dependable check is a groundedness score like 9.1, or RAGAS
faithfulness.

**Q3.** Traceable to a retrieved document: Bangladesh / Vietnam / Philippines and the
low-lying-coastline explanation (`southeast_asia_exposure`). Not traceable: the seven other
countries (India, Indonesia, Myanmar, Thailand, China, Sri Lanka, Maldives) and all their
supporting details (Odisha, Jakarta's subsidence, the Irrawaddy Delta, Bangkok,
Shanghai...). One subtle case: the Vietnam bullet mentions the Mekong Delta — that fact
exists in `CORPUS` (`mekong_delta`) but that document was NOT among the three retrieved
chunks, so relative to the context the model was given, it is still unsupported. All of
this came from the model's training knowledge: possibly true, but unverified by our
sources, which is what faithfulness measures. The fix is an output groundedness check plus
a strict "answer only from the context" prompt — exactly what `answer_question` in 9.1
does, and its answer scores 1.0.

**Q4.** `2 appel(s) · 316 tok in / 609 tok out · ≈ $0.00429`. Tokens out exceed tokens in
here because the model wrote a long 10-country answer that goes far past the context (the
Q3 problem, visible in the bill). Normally a RAG agent shows the opposite: input dominates,
since each step resends the system prompt, tool schemas, history and retrieved chunks
while the answer stays short. On a pricier model the input side is usually what grows the
bill fastest.

**Q5.** We reran this cell while preparing the lab, as the exercise asks. The three
grounded countries were stable across runs, but the extras changed — an earlier run listed
Pakistan with a "2022 floods displaced over 30 million people" figure, where the saved run
above lists the Maldives instead — and the wording changed every time. So a single run
proves nothing; you need a fixed test set (`QUESTIONS` / `GROUND_TRUTH`) and metrics
averaged over it. Our RAGAS pass on those 5 questions completed all 15 judge calls
(sequential, no rate-limit failures — the judge model is printed in the 6b output) and
scored context_recall 1.0, context_precision 1.0, faithfulness 1.0 for the production
retriever. One honest caveat: cell 6b uses the top retrieved chunk as the "answer", so
faithfulness is easy by construction there — scored on a generated answer like this
cell's, it would be much lower.


## 8. Build a real MCP server

So far your retriever is a Python function called inside `run_agent`. That works, but it
only lives inside this notebook. **MCP (Model Context Protocol)** turns your tools into a
standalone *server* that ANY MCP-compatible client (Claude Desktop, an agent framework,
the MCP Inspector) can connect to and call.

In this part you will:
1. Install the `mcp` package.
2. Write a real `mcp_server.py` with **three tools**: `web_search`, `recall_memory`, `store_finding`.
3. Launch it and verify it with a real MCP client (the Python test in 8.3), and
   optionally explore it visually in the MCP Inspector (8.4).

> **This is Deliverable 1b** — you must show 3/3 tools working before leaving.
>
> **Requirements:** `pip install mcp requests nest_asyncio`. The verification in 8.3 needs
> neither Node nor internet. `web_search` uses the internet when available and returns an
> error string when not; `recall_memory` and `store_finding` run fully locally.

In [10]:
# 8.1 — Install the MCP SDK into THIS kernel's Python (run once)
# Using {sys.executable} guarantees it installs into the same interpreter the
# notebook runs — the usual cause of 'ModuleNotFoundError: No module named mcp'.
import sys
!{sys.executable} -m pip install mcp requests nest_asyncio

# After this finishes: if the import in 8.3 still fails, RESTART THE KERNEL
# (the ↻ button at the top) and run 8.1 again, then 8.2 and 8.3.
print('Done. If 8.3 says "No module named mcp", restart the kernel and re-run.')


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: /Users/abdu07/Desktop/PGE5/AI-AGENTS-CS/part2/.venv/bin/python -m pip install --upgrade pip


Done. If 8.3 says "No module named mcp", restart the kernel and re-run.


### 8.2 — Write `mcp_server.py`

The cell below writes a complete, runnable MCP server to disk. Read the three tool
docstrings carefully — they follow the six patterns from the slides (Use when / Do NOT /
Returns / Prefer / Example / Precise naming). **The docstring is what the LLM reads to
decide when to call each tool.**

Note how each tool wraps everything in `try/except` and always returns a *string* — a tool
that raises an exception can disconnect the whole server.

In [11]:
# 8.2 — Write the MCP server to disk
mcp_server_code = '''
"""mcp_server.py — a 3-tool MCP server for climate-displacement research.
Run standalone:  python mcp_server.py
Inspect:         npx @modelcontextprotocol/inspector python mcp_server.py
"""
from mcp.server.fastmcp import FastMCP
import requests

mcp = FastMCP("research-tools")

# ── A tiny local corpus so recall_memory works offline ──
CORPUS = {
    "unhcr_2023": "According to UNHCR, 21.5 million people are displaced every year by "
                  "sudden climate events: floods, storms and droughts.",
    "worldbank_2050": "The World Bank estimates that by 2050, 216 million people could be "
                      "forced to migrate within their own country because of climate change.",
    "idmc_philippines": "The IDMC reports the Philippines saw a 15% increase in "
                        "typhoon-related displacement over the last five years.",
    "mekong_delta": "The Mekong Delta risks losing up to 40% of its floodable farmland by 2100.",
    "southeast_asia_exposure": "Bangladesh, Vietnam and the Philippines are among the most "
                               "exposed countries to climate displacement in Southeast Asia.",
}
# In-memory store for findings saved during a session
_STORE = {}


@mcp.tool()
def web_search(query: str, num_results: int = 3) -> str:
    """Search the public web for current information.

    Use when: you need facts, news or citations that are NOT already in memory.
    Do NOT use for: maths, or a topic you already saved with store_finding.
    Returns: a numbered list of results, each with a title and a snippet.
    Example: query="UNHCR climate displacement 2024"
    """
    try:
        # DuckDuckGo Instant Answer API — no key required
        r = requests.get(
            "https://api.duckduckgo.com/",
            params={"q": query, "format": "json", "no_html": 1},
            timeout=10,
        )
        r.raise_for_status()
        data = r.json()
        topics = data.get("RelatedTopics", [])
        results = []
        for t in topics[:num_results]:
            if isinstance(t, dict) and t.get("Text"):
                results.append(t["Text"])
        if not results:
            abstract = data.get("AbstractText", "")
            return abstract or "No results found. Try a broader query or use recall_memory."
        return "\\n".join(f"{i+1}. {txt}" for i, txt in enumerate(results))
    except requests.Timeout:
        return "Search timed out. Try recall_memory instead."
    except Exception as e:
        return f"Search error: {e}. Try recall_memory instead."


@mcp.tool()
def recall_memory(query: str) -> str:
    """Retrieve relevant passages from the internal knowledge base.

    Use FIRST, before web_search — it is free and instant.
    Returns: matching passages with their source id, or a message to try web_search.
    Example: query="which countries are most exposed"
    """
    try:
        q = set(query.lower().split())
        scored = []
        for doc_id, text in {**CORPUS, **_STORE}.items():
            overlap = len(q & set(text.lower().split()))
            if overlap:
                scored.append((overlap, doc_id, text))
        scored.sort(reverse=True)
        if not scored:
            return "No relevant memories. Use web_search to find new information."
        return "\\n---\\n".join(f"[{doc_id}] {text}" for _, doc_id, text in scored[:3])
    except Exception as e:
        return f"Recall error: {e}"


@mcp.tool()
def store_finding(finding: str, source: str) -> str:
    """Save a verified finding to memory so recall_memory can return it later.

    Use after web_search when you find a credible, relevant fact.
    Do NOT store: speculation or unverified claims.
    Returns: a confirmation string.
    Example: finding="Sea levels rose 3.6mm/yr", source="NASA 2023"
    """
    try:
        key = f"finding_{len(_STORE) + 1}"
        _STORE[key] = f"{finding} (source: {source})"
        return f"Stored as {key}: {_STORE[key]}"
    except Exception as e:
        return f"Store error: {e}"


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("mcp_server.py", "w") as f:
    f.write(mcp_server_code)
print("Wrote mcp_server.py")
print("Now test it two ways: the Inspector (8.3) and programmatically (8.4).")

Wrote mcp_server.py
Now test it two ways: the Inspector (8.3) and programmatically (8.4).


### 8.3 — Verify your server (required — this is the deliverable)

This test launches your `mcp_server.py` as a real subprocess and talks to it over the MCP
protocol — exactly as a real client would. It needs **no Node, no internet** (the two local
tools are enough to prove the server works) and runs anywhere Python runs.

Run it. When you see **3/3 tools working**, that is Deliverable 1b.

In [12]:
import nest_asyncio
import asyncio
import sys
import os

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

print("Notebook Python:", sys.executable)
print("Working directory:", os.getcwd())
print("mcp_server.py exists:", os.path.exists("mcp_server.py"))

async def test_mcp_server():
    params = StdioServerParameters(
        command=sys.executable,      # <-- use the notebook's Python
        args=["mcp_server.py"],
    )

    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Test 1 — list tools
            tools = await session.list_tools()
            names = [t.name for t in tools.tools]
            assert len(names) == 3, f"Expected 3 tools, got {names}"
            print(f"✓ list_tools → {names}")

            # Test 2 — recall_memory
            r = await session.call_tool(
                "recall_memory",
                {"query": "most exposed countries"},
            )
            text = r.content[0].text
            print("✓ recall_memory:", text)

            # Test 3 — store then recall
            await session.call_tool(
                "store_finding",
                {
                    "finding": "Test fact about floods",
                    "source": "test",
                },
            )

            r2 = await session.call_tool(
                "recall_memory",
                {"query": "floods test"},
            )

            print("✓ store/recall:", r2.content[0].text)

            # Test 4 — web search
            r3 = await session.call_tool(
                "web_search",
                {"query": "climate displacement 2024"},
            )

            print("✓ web_search:", r3.content[0].text[:100])

            print("\n✅ All tests passed.")

asyncio.run(test_mcp_server())

Notebook Python: /Users/abdu07/Desktop/PGE5/AI-AGENTS-CS/part2/.venv/bin/python
Working directory: /Users/abdu07/Desktop/PGE5/AI-AGENTS-CS/part2
mcp_server.py exists: True


✓ list_tools → ['web_search', 'recall_memory', 'store_finding']
✓ recall_memory: [southeast_asia_exposure] Bangladesh, Vietnam and the Philippines are among the most exposed countries to climate displacement in Southeast Asia.
✓ store/recall: [finding_1] Test fact about floods (source: test)


✓ web_search: No results found. Try a broader query or use recall_memory.

✅ All tests passed.


### 8.4 — (Optional) Explore your server visually with the MCP Inspector

The Inspector is a browser UI that lets you click each tool and see its schema — useful to
*show* how a client sees your server, but **not required** for the deliverable (8.3 already
proved it works).

**It needs Node.js.** Check with `node --version`. If you have it:

```bash
npx @modelcontextprotocol/inspector python3 mcp_server.py
```

If `npx` / `node` is **not installed**, either skip this (8.3 is sufficient) or install Node:

```bash
# macOS:            brew install node
# Windows/Linux:    download from https://nodejs.org  (LTS)
```

Then in the browser tab that opens:
1. Confirm all **3 tools** appear: `web_search`, `recall_memory`, `store_finding`.
2. Run `recall_memory` with `query = "most exposed countries"` → you get passages back.
3. Run `store_finding`, then `recall_memory` it back to see the round-trip.

### 📝 Questions on your MCP server

**Q1 — Why a server, not a function?** Your retriever worked fine as a Python function in
Part 7. What does wrapping it as an MCP *server* let you do that a function cannot?

**Q2 — Read your own docstrings.** Open `mcp_server.py`. For `web_search`, point to the exact
sentence that tells the LLM *when NOT* to call it. Why does removing that line make the agent
call `web_search` more often than it should?

**Q3 — Error handling.** Each tool returns a string even on failure. Trace what would happen to
the whole agent if `web_search` raised an exception instead of returning an error string.

**Q4 — Test coverage.** The programmatic test checks `recall_memory` and the store→recall
round-trip, but only lists `web_search`. Why might a test suite deliberately avoid *calling*
`web_search` in CI, and what would you mock instead?

### Answers — MCP server

**Q1.** A function only exists inside this notebook's Python process. The MCP server is a
separate process that speaks a standard protocol (JSON-RPC over stdio), so any MCP client
— Claude Desktop, another agent framework, the Inspector, someone else's script — can list
and call the same three tools without importing our code. It also lives independently of
the agent: it can be restarted, deployed or permission-scoped on its own.

**Q2.** The sentence is: "Do NOT use for: maths, or a topic you already saved with
store_finding." The LLM picks tools from docstrings alone. Without that negative line, web
search looks applicable to almost everything, including questions `recall_memory` already
answers for free, so the agent starts calling it too often.

**Q3.** With FastMCP specifically, an exception inside a tool handler is caught by the
framework and sent back to the client as a generic tool error, so the process would not
actually die — but the model would only see an unhelpful error blob. Returning our own
string keeps the failure actionable: "Search timed out. Try recall_memory instead." tells
the model exactly what to do next. And exceptions outside a tool handler (during startup
or transport handling) do kill the process, close the stdio pipe and disconnect the
session. Hence the rule: never let an exception escape, always return a string.

**Q4.** `web_search` depends on an external service, so calling it in CI would be slow and
flaky (rate limits, outages), and a red build would say nothing about our code. Our run
shows the nuance: the HTTP call to DuckDuckGo succeeded (no exception, status OK) but the
response had no topics and no abstract for that query, so the server returned its own
fallback string "No results found. Try a broader query or use recall_memory." — the code
path worked, the external data was just empty. In CI you would mock the HTTP layer (patch
`requests.get` to return canned JSON) and test the formatting and the error paths (timeout
gives a string, HTTP error gives a string), which is what actually needs proving.


## 9. Exercises

> Try the `# TODO` cell, then expand the solution.

### 9.1 — Add a groundedness filter

Reuse Guillaume's `groundedness` idea and integrate it into an answer function: if the
score is below 0.5, add a warning to the answer.


In [13]:
# 9.1 — Groundedness filter: warn when the answer drifts from its contexts.
import time

def check_groundedness(answer: str, contexts: list) -> tuple:
    """Share of the answer's meaningful words traceable to the retrieved contexts.
    Returns (score in [0,1], label). Same idea as RAGAS faithfulness, zero-dependency."""
    ctx_words = set(w for c in contexts for w in _tokenise(c))
    ans_words = [w for w in _tokenise(answer) if len(w) > 3]
    if not ans_words:
        return 0.0, "empty answer"
    score = sum(w in ctx_words for w in ans_words) / len(ans_words)
    label = "\u2705 grounded" if score >= 0.5 else "\u26a0\ufe0f  possibly hallucinated"
    return round(score, 2), label


def answer_question(query: str, k: int = 3) -> str:
    """RAG answer with a groundedness gate: score < 0.5 -> visible warning."""
    contexts = production_retrieve(query, k_final=k)
    client = demo([{"final": "Bangladesh, Vietnam and the Philippines are among the most "
                             "exposed countries, due to their low-lying coastlines."}])
    messages = [
        {"role": "system", "content": "Answer ONLY from the provided context, in one sentence."},
        {"role": "user", "content": "Context:\n" + "\n---\n".join(contexts)
                                    + f"\n\nQuestion: {query}"},
    ]
    for attempt in range(5):          # free-tier keys rate-limit at ~1 req/s
        try:
            msg = client.complete(messages)
            break
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                time.sleep(5 * (attempt + 1))
            else:
                raise
    else:
        raise RuntimeError("still rate-limited after 5 attempts")
    answer = msg.content or ""
    score, label = check_groundedness(answer, contexts)
    out = f"{answer}\n[groundedness {score} {label}]"
    if score < 0.5:
        out += "\n\u26a0\ufe0f WARNING: this answer is weakly supported by the retrieved documents."
    return out

q = QUESTIONS[1]
print(f"Q: {q}\n")
print(answer_question(q))

# Sanity check: a fabricated claim must trip the warning.
ctx = production_retrieve(q, k_final=3)
fabricated = "Atlantis will relocate three billion citizens to the Moon colonies by 2030."
print("\nFabricated claim ->", check_groundedness(fabricated, ctx))


Q: Which countries are most exposed along Southeast Asian coastlines?



Bangladesh, Vietnam, and the Philippines are among the countries most exposed to climate displacement in Southeast Asia due to their long low-lying coastlines and dense coastal populations.
[groundedness 1.0 ✅ grounded]

Fabricated claim -> (0.0, '⚠️  possibly hallucinated')


<details><summary>✅ Solution 9.1</summary>

```python
def check_groundedness(answer: str, contexts: list) -> tuple:
    ctx_words = set(w for c in contexts for w in _tokenise(c))
    ans_words = [w for w in _tokenise(answer) if len(w) > 3]
    if not ans_words: return 0.0, "empty answer"
    score = sum(w in ctx_words for w in ans_words) / len(ans_words)
    label = "✅ grounded" if score >= 0.5 else "⚠️  possibly hallucinated"
    return round(score, 2), label
```
</details>


### 9.2 — Metadata and filtering

Enrich the index with metadata (source, year) and modify `hybrid_retrieve` to accept
only documents from a given source.


In [14]:
# 9.2 — Metadata + filtering: hybrid retrieval restricted to a given source / year.

META = {
    "unhcr_2023":              {"source": "UNHCR",      "year": 2023},
    "worldbank_2050":          {"source": "World Bank", "year": 2021},
    "idmc_philippines":        {"source": "IDMC",       "year": 2023},
    "mekong_delta":            {"source": "IPCC",       "year": 2022},
    "southeast_asia_exposure": {"source": "UNHCR",      "year": 2022},
    "climate_finance":         {"source": "GCF",        "year": 2023},
    "cop_negotiations":        {"source": "UNFCCC",     "year": 2022},
    "refugee_definition":      {"source": "UNHCR",      "year": 1951},
    "drought_agriculture":     {"source": "FAO",        "year": 2023},
    "sea_level_cities":        {"source": "World Bank", "year": 2022},
    "typhoon_intensity":       {"source": "IPCC",       "year": 2023},
    "economic_migration":      {"source": "ILO",        "year": 2022},
}

def index_with_meta(docs_dict: dict, meta_dict: dict):
    """Build a hybrid (dense + BM25 + RRF) retriever whose search() accepts
    metadata filters: source=... and/or year_min=... Pre-filter, then search —
    filtering after retrieval could leave fewer than k results."""
    text_to_id = {t: n for n, t in docs_dict.items()}

    def search(query: str, k: int = 3, source: str = None, year_min: int = None) -> list:
        allowed = {
            doc_id: text for doc_id, text in docs_dict.items()
            if (source is None or meta_dict.get(doc_id, {}).get("source") == source)
            and (year_min is None or meta_dict.get(doc_id, {}).get("year", 0) >= year_min)
        }
        if not allowed:
            return []
        dense = TinyTfidf(allowed).search(query, k=k * 2)
        bm25  = TinyBM25(allowed).search(query, k=k * 2)
        fused = rrf_fusion([dense, bm25])
        return [(text_to_id[text], meta_dict.get(text_to_id[text], {}), text)
                for text, _ in fused[:k]]

    return search

search_meta = index_with_meta(CORPUS, META)

q = "How many people are displaced by climate events?"
print(f"Query: {q!r}\n")
print("No filter:")
for doc_id, m, text in search_meta(q, k=2):
    print(f"  [{doc_id} | {m['source']} {m['year']}] {text[:70]}\u2026")
print("\nsource='World Bank' only:")
for doc_id, m, text in search_meta(q, k=2, source="World Bank"):
    print(f"  [{doc_id} | {m['source']} {m['year']}] {text[:70]}\u2026")
print("\nsource='UNHCR', year_min=2022:")
for doc_id, m, text in search_meta(q, k=2, source="UNHCR", year_min=2022):
    print(f"  [{doc_id} | {m['source']} {m['year']}] {text[:70]}\u2026")


Query: 'How many people are displaced by climate events?'

No filter:
  [unhcr_2023 | UNHCR 2023] According to UNHCR, 21.5 million people are displaced every year by su…
  [refugee_definition | UNHCR 1951] The 1951 Geneva Convention does not recognise 'climate refugees' as a …

source='World Bank' only:
  [worldbank_2050 | World Bank 2021] The World Bank estimates that by 2050, 216 million people could be for…
  [sea_level_cities | World Bank 2022] Coastal cities such as Jakarta, Miami and Alexandria are investing in …

source='UNHCR', year_min=2022:
  [unhcr_2023 | UNHCR 2023] According to UNHCR, 21.5 million people are displaced every year by su…
  [southeast_asia_exposure | UNHCR 2022] Bangladesh, Vietnam and the Philippines are among the countries most e…


## ✅ Recap

- **TF-IDF baseline** → measurable starting point with the hit@k / MRR proxy.
- **Parent-child chunking** → retrieve small, return large: ↑ recall AND precision.
- **Hybrid BM25 + TF-IDF + RRF** → catches the exact terms semantics misses. ↑ recall.
- **Cross-encoder reranking** → cross query×document relevance. ↑ precision (MRR).
- **The metric** tells you *which* component to improve first.

➡️ **Lab 2**: we secure the agent — attack surface, L1/L4 filters, token budget.
